# Untuk mendeteksi kesalahan ejaan (typo) di dalam sebuah kalimat sekaligus memperbaikinya menggunakan NLP

In [3]:
import string
import difflib

# 1. Menyiapkan Kamus Kata Baku sebagai referensi program
# Dalam aplikasi nyata, kamus ini bisa berisi puluhan ribu kata dari KBBI
kamus_baku = [
    "saya", "sedang", "belajar", "pemrosesan", "bahasa", "alami", "kecerdasan","adalah",
    "hari", "ini", "sangat", "seru", "dan", "menantang", "komputer"
]

sample_text = (
        "Perosesan bahassa alami adalah cabang kecerdassan buatan yang sangat penting. "
        "Komputer dapat memanipulasi teks dan mengekstrak wawasan bermakna. "
        "Saya belahar NPL dan itu menyenagkan. "
        "Binus adlah lembaga pendidikan dibidang komputer."
    )
def perbaiki_ejaan_kalimat(kalimat_input):
    # Tahap 1: Normalisasi & Tokenisasi sederhana
    # Mengubah ke huruf kecil dan menghapus tanda baca agar fokus pada kata
    kalimat_clean = kalimat_input.lower().translate(str.maketrans('', '', string.punctuation))
    kata_list = kalimat_clean.split()
    
    kata_terkoreksi = []
    daftar_typo = []
    
    # Tahap 2: Deteksi dan Koreksi per Kata
    for kata in kata_list:
        # Jika kata ada di kamus, artinya ejaannya sudah benar
        if kata in kamus_baku:
            kata_terkoreksi.append(kata)
        else:
            # Jika tidak ada, cari kata paling mirip di kamus menggunakan Levenshtein/Edit Distance
            pilihan_terdekat = difflib.get_close_matches(kata, kamus_baku, n=1, cutoff=0.5)
            
            if pilihan_terdekat:
                kata_terkoreksi.append(pilihan_terdekat[0])
                daftar_typo.append(f"'{kata}' -> '{pilihan_terdekat[0]}'")
            else:
                # Jika terlalu jauh berbeda dan tidak ada yang mirip, biarkan kata aslinya
                kata_terkoreksi.append(kata)
    
    # Menyatukan kembali token-token kata menjadi kalimat utuh
    kalimat_hasil = " ".join(kata_terkoreksi)
    
    # Menampilkan Hasil Analysis
    print(f"Kalimat Asli    : {kalimat_input}")
    print(f"Deteksi Typo    : {', '.join(daftar_typo) if daftar_typo else 'Tidak ada typo'}")
    print(f"Hasil Perbaikan : {kalimat_hasil}")
    print("-" * 50)

# --- Uji Coba Program ---
# Contoh 1: Kalimat dengan beberapa typo (salah ketik)
perbaiki_ejaan_kalimat(sample_text)

# Contoh 2: Kalimat yang sudah benar sejak awal
perbaiki_ejaan_kalimat("Hari ini sangat seru dan menantang!")

Kalimat Asli    : Perosesan bahassa alami adalah cabang kecerdassan buatan yang sangat penting. Komputer dapat memanipulasi teks dan mengekstrak wawasan bermakna. Saya belahar NPL dan itu menyenagkan. Binus adlah lembaga pendidikan dibidang komputer.
Deteksi Typo    : 'perosesan' -> 'pemrosesan', 'bahassa' -> 'bahasa', 'cabang' -> 'menantang', 'kecerdassan' -> 'kecerdasan', 'buatan' -> 'menantang', 'yang' -> 'sedang', 'penting' -> 'menantang', 'dapat' -> 'sangat', 'wawasan' -> 'bahasa', 'bermakna' -> 'belajar', 'belahar' -> 'belajar', 'menyenagkan' -> 'menantang', 'binus' -> 'ini', 'adlah' -> 'adalah', 'pendidikan' -> 'sedang', 'dibidang' -> 'sedang'
Hasil Perbaikan : pemrosesan bahasa alami adalah menantang kecerdasan menantang sedang sangat menantang komputer sangat memanipulasi teks dan mengekstrak bahasa belajar saya belajar npl dan itu menantang ini adalah lembaga sedang sedang komputer
--------------------------------------------------
Kalimat Asli    : Hari ini sangat seru dan m

# Kamus bahasa indo - eng 

In [6]:
import nltk
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 1. Unduh resource yang diperlukan untuk Lemmatizer Bahasa Inggris
nltk.download('wordnet')
nltk.download('omw-1.4')

# 2. Inisialisasi komponen NLP (Stemmer Indonesia & Lemmatizer Inggris)
factory = StemmerFactory()
stemmer_id = factory.create_stemmer()
lemmatizer_en = WordNetLemmatizer()

# 3. Database Kamus Sederhana (Kata Dasar / Lemma)
kamus_id_to_en = {
    "makan": "eat",
    "jalan": "walk",
    "buku": "book",
    "belajar": "study",
    "tidur": "sleep"
}

# Membuat kamus kebalikannya (Inggris ke Indonesia) secara otomatis
kamus_en_to_id = {v: k for k, v in kamus_id_to_en.items()}

# 4. Fungsi Penerjemah Cerdas dengan NLP
def cari_di_kamus(kata_input, bahasa_asal="id"):
    # Tahap Normalisasi: Mengubah ke huruf kecil
    kata_clean = kata_input.strip().lower()

    print(f"Kata yang dicari: '{kata_input}'")

    if bahasa_asal == "id":
        # NLP Tahap 1: Stemming Bahasa Indonesia (misal: 'memakan' -> 'makan')
        kata_dasar = stemmer_id.stem(kata_clean)
        print(f"Proses NLP (Kata Dasar ID): {kata_dasar}")

        # Pencarian di database kamus
        hasil = kamus_id_to_en.get(kata_dasar, "Kata tidak ditemukan di kamus.")
        print(f"Arti (Inggris): {hasil}")

    elif bahasa_asal == "en":
        # NLP Tahap 2: Lemmatization Bahasa Inggris (misal: 'books' -> 'book', 'eating' -> 'eat')
        # Kita cek sebagai kata benda ('n') dan kata kerja ('v')
        lemma_noun = lemmatizer_en.lemmatize(kata_clean, pos='n')
        lemma_verb = lemmatizer_en.lemmatize(kata_clean, pos='v')

        # Cari mana lemma yang cocok di kamus data kita
        if lemma_noun in kamus_en_to_id:
            kata_dasar = lemma_noun
        else:
            kata_dasar = lemma_verb

        print(f"Proses NLP (Kata Dasar EN): {kata_dasar}")

        # Pencarian di database kamus
        hasil = kamus_en_to_id.get(kata_dasar, "Word not found in dictionary.")
        print(f"Arti (Indonesia): {hasil}")

    print("-" * 40)

# --- Uji Coba Program Kamus ---

print("--- UJI COBA KAMUS INDONESIA -> INGGRIS ---")
# Pengguna memasukkan kata berimbuhan Indonesia
cari_di_kamus("Memakan", bahasa_asal="id")
cari_di_kamus("belajar", bahasa_asal="id") # 'belajar'

print("\n--- UJI COBA KAMUS INGGRIS -> INDONESIA ---")
# Pengguna memasukkan kata jamak (plural) atau berimbuhan Inggris
cari_di_kamus("books", bahasa_asal="en")
cari_di_kamus("sleeping", bahasa_asal="en")

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


--- UJI COBA KAMUS INDONESIA -> INGGRIS ---
Kata yang dicari: 'Memakan'
Proses NLP (Kata Dasar ID): makan
Arti (Inggris): eat
----------------------------------------
Kata yang dicari: 'belajar'
Proses NLP (Kata Dasar ID): ajar
Arti (Inggris): Kata tidak ditemukan di kamus.
----------------------------------------

--- UJI COBA KAMUS INGGRIS -> INDONESIA ---
Kata yang dicari: 'books'
Proses NLP (Kata Dasar EN): book
Arti (Indonesia): buku
----------------------------------------
Kata yang dicari: 'sleeping'
Proses NLP (Kata Dasar EN): sleep
Arti (Indonesia): tidur
----------------------------------------


# Menterjamah per Kalimat indo -eng
Pustaka Python deep-translator atau googletrans yang berbasis NLP untuk melakukan penerjemahan kalimat dengan memperhatikan konteks.Pustaka ini menganalisis struktur seluruh kalimat (seperti subjek, predikat, objek) sehingga susunan tata bahasa (grammar) hasil terjemahan di bahasa Inggris tetap natural dan benar.

In [10]:
from deep_translator import GoogleTranslator

def terjemahkan_kalimat(kalimat_indo):
    print(f"Kalimat Asli (Indonesia) : {kalimat_indo}")
    
    try:
        # Menggunakan NLP Translator dengan menentukan bahasa asal (id) dan tujuan (en)
        hasil_terjemahan = GoogleTranslator(source='id', target='en').translate(kalimat_indo)
        
        print(f"Hasil Terjemahan (Inggris): {hasil_terjemahan}")
    except Exception as e:
        print(f"Terjadi kesalahan saat memproses teks: {e}")
        
    print("-" * 50)

# --- Uji Coba Program Penerjemah ---

# Contoh 1: Kalimat percakapan sehari-hari
terjemahkan_kalimat("Saya sedang belajar pemrosesan bahasa alami hari ini.")

# Contoh 2: Kalimat dengan kata berimbuhan kompleks untuk melihat kecerdasan konteksnya
terjemahkan_kalimat("Mereka memotong biaya produksi untuk mempertahankan kestabilan perusahaan.")

Kalimat Asli (Indonesia) : Saya sedang belajar pemrosesan bahasa alami hari ini.
Hasil Terjemahan (Inggris): I'm learning natural language processing today.
--------------------------------------------------
Kalimat Asli (Indonesia) : Mereka memotong biaya produksi untuk mempertahankan kestabilan perusahaan.
Hasil Terjemahan (Inggris): They cut production costs to maintain company stability.
--------------------------------------------------
